In [1]:
path = '/Order Details.csv'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install tabula-py pandas sentence-transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.3 MB/s eta 0:00:00


In [4]:
import tabula
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [6]:
GEMINI_API = userdata.get("Devesh_API")
if GEMINI_API is None:
  raise ValueError("No GEMINI_API found in Colab Secrets")
genai.configure(api_key=GEMINI_API)

In [9]:
pdf_path = '/Order Details.csv'
df = pd.read_csv(pdf_path)
print(df.head())
df

  Order ID  Amount  Profit  Quantity     Category      Sub-Category
0  B-25601  1275.0 -1148.0         7    Furniture         Bookcases
1  B-25601    66.0   -12.0         5     Clothing             Stole
2  B-25601     8.0    -2.0         3     Clothing       Hankerchief
3  B-25601    80.0   -56.0         4  Electronics  Electronic Games
4  B-25602   168.0  -111.0         2  Electronics            Phones


,Order ID,Amount,Profit,Quantity,Category,Sub-Category
0,B-25601,1275.0,-1148.0,7,Furniture,Bookcases
1,B-25601,66.0,-12.0,5,Clothing,Stole
2,B-25601,8.0,-2.0,3,Clothing,Hankerchief
3,B-25601,80.0,-56.0,4,Electronics,Electronic Games
4,B-25602,168.0,-111.0,2,Electronics,Phones
...,...,...,...,...,...,...
1495,B-26099,835.0,267.0,5,Electronics,Phones
1496,B-26099,2366.0,552.0,5,Clothing,Trousers
1497,B-26100,828.0,230.0,2,Furniture,Chairs
1498,B-26100,34.0,10.0,2,Clothing,T-shirt


In [10]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [13]:
documents = []
for _, row in df.iterrows():
  doc = (
      f"Order ID: {row['Order ID']}.\n"
      f"Amount: {row['Amount']}.\n"
      f"Profit: {row['Profit']}.\n"
      f"Quantity: {row['Quantity']}.\n"
      f"Category: {row['Category']}.\n"
      f"Sub-Category: {row['Sub-Category']}."
   )
  documents.append(doc)

print(documents)

['Order ID: B-25601.\nAmount: 1275.0.\nProfit: -1148.0.\nQuantity: 7.\nCategory: Furniture.\nSub-Category: Bookcases.', 'Order ID: B-25601.\nAmount: 66.0.\nProfit: -12.0.\nQuantity: 5.\nCategory: Clothing.\nSub-Category: Stole.', 'Order ID: B-25601.\nAmount: 8.0.\nProfit: -2.0.\nQuantity: 3.\nCategory: Clothing.\nSub-Category: Hankerchief.', 'Order ID: B-25601.\nAmount: 80.0.\nProfit: -56.0.\nQuantity: 4.\nCategory: Electronics.\nSub-Category: Electronic Games.', 'Order ID: B-25602.\nAmount: 168.0.\nProfit: -111.0.\nQuantity: 2.\nCategory: Electronics.\nSub-Category: Phones.', 'Order ID: B-25602.\nAmount: 424.0.\nProfit: -272.0.\nQuantity: 5.\nCategory: Electronics.\nSub-Category: Phones.', 'Order ID: B-25602.\nAmount: 2617.0.\nProfit: 1151.0.\nQuantity: 4.\nCategory: Electronics.\nSub-Category: Phones.', 'Order ID: B-25602.\nAmount: 561.0.\nProfit: 212.0.\nQuantity: 3.\nCategory: Clothing.\nSub-Category: Saree.', 'Order ID: B-25602.\nAmount: 119.0.\nProfit: -5.0.\nQuantity: 8.\nCatego

In [14]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(documents, convert_to_numpy=True, show_progress_bar=True)
print(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

[[-0.01090907  0.07967411 -0.04885895 ... -0.08077276 -0.0411204
   0.00027728]
 [-0.0751306   0.1300372  -0.02196351 ... -0.12065041 -0.03797528
  -0.03173365]
 [-0.12288737  0.11469384 -0.05307913 ... -0.07987432 -0.02923615
   0.00411705]
 ...
 [-0.00275785  0.04942099 -0.04857961 ... -0.07909759 -0.03407723
  -0.03529269]
 [-0.04622171  0.11902051 -0.02431773 ... -0.1514779  -0.06097862
  -0.07088423]
 [-0.04086391  0.1206338  -0.03599045 ... -0.14710647 -0.06854588
  -0.06180749]]


In [15]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
faiss.write_index(index, 'faiss_index.bin')

In [16]:
def retieve_context(query, k=3):
  query_embedding = embedding_model.encode([query])
  distance, indices = index.search(query_embedding, k)
  return "\n".join([documents[i] for i in indices[0]])

In [17]:
generation_config = {
    "temperature": 0.4,
    "max_output_tokens": 512
}
print(generation_config)
gemini_model = genai.GenerativeModel(model_name='models/gemini-2.5-flash',generation_config=generation_config)

{'temperature': 0.4, 'max_output_tokens': 512}


In [18]:
chat_history = []
def chat_with_bot(user_input):
  global chat_history

  context = retieve_context(user_input)
  prompt = f"""
You are a helpful conversational data analyst assistant. Please refer to the context below and answer the user's question.
Context:
{context}
User's Question:
{user_input}

Rules:
- Be Conversational
- Answer only using context
- If you don't know the answer, say you don't have enough information
"""
  response = gemini_model.generate_content(prompt)
  answer = response.candidates[0].content.parts[0].text
  chat_history.append({"user": user_input, "bot": answer})
  return answer

In [19]:
print("Order Anatytics Chat Bot Ready !!!")
print("Type 'exit' to stop\n")

while True:
  user_input = input("User: ")
  if user_input.lower() in ['exit', 'quit', 'bye']:
    print("Goodbye!!!")
    break
  response = chat_with_bot(user_input)
  print(f"Bot: {response}")
  print("-"*60)

Order Anatytics Chat Bot Ready !!!
Type 'exit' to stop

User: what is the total amount
Bot: Based on the information you've provided, the total amount is 563.0.
------------------------------------------------------------
User: exit
Goodbye!!!
